In [ ]:
import geemap
import ee 
import os

In [ ]:
geemap.ee_initialize()

In [ ]:
Map = geemap.Map()

In [ ]:
# Define the Relative Path
# '..' means go up one folder from where this notebook is saved
file_path = os.path.join('..', 'data', 'processed', 'Dholera_Taluk.geojson')

In [ ]:
roi = geemap.geojson_to_ee(file_path)
Map.addLayer(roi, {'color': 'blue', 'width': 2}, 'Dholera Taluka Boundary')
Map.centerObject(roi, 10)
Map

This code analyzes nighttime lights captured by the Suomi Satellite around 1:30 AM to assess economic activity in a region.

In [ ]:
# 1. Load the VIIRS Nighttime Lights ImageCollection
# We filter by date; here we'll take a recent monthly composite or annual mean
import geemap.colormaps as cm
viirs = ee.ImageCollection("NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG") \
    .filterDate('2025-10-01', '2026-03-31') \
    .select('avg_rad') \
    .median() # Taking the median to reduce noise/cloud artifacts
#post-monsoon (2025) data to very recent March(2026) data

# 2. Clip the data to your Dholera Taluka boundary
nightlight_roi = viirs.clip(roi)

# 3. Define visualization parameters
# 'avg_rad' (radiance) usually ranges from 0 to ~100 in urban areas
# 'inferno' or 'magma' are great for nighttime lights
palette = cm.get_palette('inferno', n_class=8) 

viz_params = {
    'min': 0,
    'max': 20,
    'palette': palette
}

# 4. Add to the map you already initialized
Map.addLayer(nightlight_roi, viz_params, 'VIIRS Nightlights')
Map

This code section, which is identical to the one used in RQ1, calculates the regional build-up and produces the output: build_up_mask for 2025 (specifically, using post-monsoon data).

In [ ]:
# Load and Filter Sentinel-2 Surface Reflectance (2026)
# We use the Harmonized collection for consistency
s2 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
    .filterBounds(roi) \
    .filterDate('2025-10-01', '2025-12-31') \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 5)) \
    .median() \
    .clip(roi)

In [ ]:
# 2. Define Index Functions
def calculate_indices(image):
    # Band aliases for Sentinel-2
    # B11=SWIR1, B8=NIR, B3=Green, B4=Red
    res = image.expression(
        '(SWIR1 - NIR) / (SWIR1 + NIR)', {
            'SWIR1': image.select('B11'),
            'NIR': image.select('B8')
        }).rename('NDBI')
    
    mndwi = image.expression(
        '(Green - SWIR1) / (Green + SWIR1)', {
            'Green': image.select('B3'),
            'SWIR1': image.select('B11')
        }).rename('MNDWI')
    
    savi = image.expression(
        '((NIR - Red) * 1.5) / (NIR + Red + 0.5)', {
            'NIR': image.select('B8'),
            'Red': image.select('B4')
        }).rename('SAVI')
        
    return image.addBands([res, mndwi, savi])

# Apply the indices
processed_img = calculate_indices(s2)

In [ ]:
# 3. Create the "Clean" Built-up Mask (The Logic)
# Thresholds: 
# NDBI > 0 (Built-up typically positive)
# MNDWI < 0 (Exclude water/moist salt pans)
# SAVI < 0.2 (Exclude vegetation)
built_up_mask = processed_img.expression(
    '(NDBI > 0.05) && (MNDWI < 0) && (SAVI < 0.18)', {
        'NDBI': processed_img.select('NDBI'),
        'MNDWI': processed_img.select('MNDWI'),
        'SAVI': processed_img.select('SAVI')
    }).rename('BuiltUp')

In [ ]:
# 4. Visualization
Map = geemap.Map()
Map.centerObject(roi, 11)

# True Color for Reference
Map.addLayer(s2, {'bands': ['B4', 'B3', 'B2',], 'min': 0, 'max': 3000}, 'S2 True Color (2025)')

# The Result: 1 = White (Built-up), 0 = Black (Everything Else)
Map.addLayer(built_up_mask, {'min': 0, 'max': 1, 'palette': ['black', 'white']}, 'Clean Built-up Layer')

Map

VIIRS Data: Fixed at SAVI >= 0.6.
Purpose: This lower threshold effectively captures industrial activity and major infrastructure like airports (which often have dim nightlight signatures despite being significant built-up features). 

"Under Construction" Areas (Soil Disturbance) \
Logic: Identified as an add-on over the build-up mask, focusing on disturbed soil.\
Conditions: is_soil_disturbed is true when the following conditions are met simultaneously:\
0.18 < SAVI < 0.3 (Indicates sparse vegetation/exposed soil characteristic of disturbed land)\
NDBI > 0.05 (Confirms presence of built-up/developing characteristics)\
MNDWI < -0.4 (Strictly masks out salt pan lands, ensuring focus on genuine soil disturbance)\

In [ ]:
# ============================================================
# RQ2 — Cluster Typology Classification Map
# Classes: Active Industrial / Dormant-Speculative /
#          Under-Construction / Salt Pan (False Positive)
# ============================================================
#
# Variables assumed to exist from RQ1 / earlier in this notebook:
#   s2             → Sentinel-2 median composite (Oct–Dec 2025)
#   built_up_mask  → Binary mask: 1 = confirmed built-up (2025)
#   nightlight_roi → VIIRS clipped to roi (median, Oct 2025–Mar 2026)
#   roi            → Dholera Taluka boundary (EE geometry)
# ============================================================

# ── Step 1: Recompute individual spectral layers ─────────────
ndbi  = s2.normalizedDifference(['B11', 'B8']).rename('NDBI')
mndwi = s2.normalizedDifference(['B3',  'B11']).rename('MNDWI')
savi  = s2.expression(
    '((NIR - RED) * 1.5) / (NIR + RED + 0.5)',
    {'NIR': s2.select('B8'), 'RED': s2.select('B4')}
).rename('SAVI')

# ── Step 2: VIIRS to Sentinel-2 resolution ──────────
# NTL: use a LOW threshold since most of Dholera is ~0.6 background
is_active_night = nightlight_roi.gt(0.6)

# ── Step 3: Calibrated masks ────────────────────────────────
# Under-Construction:
# The green flood was SAVI catching all arid background
# Now requires ALL THREE: SAVI disturbance + NDBI signal + not water
is_soil_disturbed = savi.gt(0.18) \
    .And(savi.lt(0.3)) \
    .And(ndbi.gt(0.05)) \
    .And(mndwi.lt(-0.4)) \
  #  .And(built_up_mask.eq(0))
 # ← must have built-up signal too
 # ← exclude any water/salt influence

# ── Step 4: Classification (unchanged) ───────────────────────
cluster_map = ee.Image(0) \
    .where(is_soil_disturbed,                                  3) \
    .where(built_up_mask.eq(1).And(is_active_night.Not()),     2) \
    .where(built_up_mask.eq(1).And(is_active_night),           1) \
    .clip(roi) \
    .rename('ClusterType')


# ── Step 5: Visualize (built_up_mask layer REMOVED) ──────────
cluster_palette = {
        'min': 0,
        'max': 3,
        'palette': [
            '1a1a1a',   # 0 → Dark   = Background\n",
            'e63946',   # 1 → Red    = Active Industrial\n",
            '6a0572',   # 2 → Purple = Dormant / Speculative\n",
            'f4a261',   # 3 → Orange = Under-Construction\n",
            ]}
Map_Cluster = geemap.Map()
Map_Cluster.centerObject(roi, 11)

Map_Cluster.addLayer(
    s2, {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000},
    'S2 True Colour (2025)', shown=False
)
Map_Cluster.addLayer(
    nightlight_roi,
    {'min': 0, 'max': 20, 'palette': ['000000', 'ffff99']},
    'VIIRS Nightlights (ref)', shown=False
)

Map_Cluster.addLayer(cluster_map, cluster_palette, 'Cluster Typology (RQ2)')
# Map_Cluster.addLayer(roi, {'color': '00ffcc', 'width': 2}, 'Dholera Taluka')

legend_dict = {
    '⬛ Background':                   '1a1a1a',
    '🔴 Active Industrial':            'e63946',
    '🟣 Dormant / Speculative':        '6a0572',
    '🟠 Under-Construction':           'f4a261',
}
Map_Cluster.add_legend(
    title='Cluster Typology - Dholera (RQ2)',
    legend_dict=legend_dict,
    position='bottomright'
)
Map_Cluster.addLayer(roi.style(color='00ffcc', fillColor='00000000', width=2), {}, 'Dholera Boundary')

print("Displaying Cluster Typology Map...")
display(Map_Cluster)

In [ ]:
# ============================================================
# Cluster Typology — Area Statistics + IUR (Optimised)
# ============================================================

pixel_area = ee.Image.pixelArea().rename('area')

# ClusterType must be band 0 (groupField), pixel_area must be band 1 (the value)
# So: cluster_map first, then addBands(pixel_area)
area_by_class = pixel_area \
    .addBands(cluster_map) \
    .reduceRegion(
        reducer=ee.Reducer.sum().group(
            groupField=1,       # band index 1 = ClusterType
            groupName='class'
        ),
        geometry=roi,
        scale=10,
        maxPixels=1e13
    ).getInfo()

# Parse grouped result into {class_val: area_km2}
area_dict = {
    int(g['class']): round(g['sum'] / 1e6, 3)
    for g in area_by_class['groups']
}

a_active       = area_dict.get(1, 0)
a_dormant      = area_dict.get(2, 0)
a_construction = area_dict.get(3, 0)
total_builtup  = a_active + a_dormant + a_construction

print("=" * 58)
print("    INDUSTRIAL CLUSTER TYPOLOGY — AREA REPORT (RQ2)")
print("=" * 58)
print(f"  🔴 Active Industrial        : {a_active:>9.3f} km²  ({a_active/total_builtup*100:>5.1f}%)")
print(f"  🟣 Dormant / Speculative    : {a_dormant:>9.3f} km²  ({a_dormant/total_builtup*100:>5.1f}%)")
print(f"  🟠 Under-Construction       : {a_construction:>9.3f} km²  ({a_construction/total_builtup*100:>5.1f}%)")
print("-" * 58)
print(f"  📦 Total Built-up (IUR base): {total_builtup:>9.3f} km²")
print("=" * 58)

iur = round(a_active / total_builtup * 100, 2) if total_builtup > 0 else 0
print(f"\n  📈 Industrial Utilisation Ratio (IUR): {iur}%")
print(f"  → Only {iur}% of built-up land shows confirmed economic activity.")
print(f"  → {round(a_dormant/total_builtup*100,1)}% is paved but completely dark → 'Ghost Grid' pattern.")
print(f"  → {round(a_construction/total_builtup*100,1)}% shows active soil disturbance → pipeline of growth.\n")